# matba — the press, in Colab

Posters in → book out, sealed, pushed, minted. Same engine as the local app; no local server needed.

**The gates are unchanged.** Nothing pushes or mints until you type the word into the prompt.
Tokens are read from a file you provide and are never written into the notebook or a repo.

In [ ]:
#@title 1 · Install the press  { display-mode: "form" }
# stdlib only — nothing is pip-installed. We just fetch matba.py and check the tools we drive.
import os, subprocess, textwrap, shutil, json, pathlib

MATBA_URL = "https://raw.githubusercontent.com/zistgah/matba/main/matba.py"  #@param {type:"string"}
if not os.path.exists("matba.py"):
    try:
        import urllib.request; urllib.request.urlretrieve(MATBA_URL, "matba.py")
        print("fetched matba.py")
    except Exception as e:
        print("could not fetch (%s) — upload matba.py with the file browser on the left" % e)
os.environ["MATBA_HOME"] = "/content/matba-projects"

for t in ("git", "gh", "misty"):
    print("%-6s %s" % (t, shutil.which(t) or "ABSENT — declared, not worked around"))
print("\ngh and misty are absent on a fresh Colab. Cells 2-5 work without them;")
print("push needs gh, mint needs misty. Cell 6 installs both if you want the full path here.")

In [ ]:
#@title 2 · Mount Drive and point at your posters  { display-mode: "form" }
USE_DRIVE = True  #@param {type:"boolean"}
POSTER_DIR = "/content/drive/MyDrive/plates"  #@param {type:"string"}

if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
import os
print(POSTER_DIR, "->", len([f for f in os.listdir(POSTER_DIR)
      if f.lower().endswith((".png",".jpg",".jpeg",".webp"))]) if os.path.isdir(POSTER_DIR) else "NOT FOUND")

In [ ]:
#@title 3 · Create the book and take the plates in  { display-mode: "form" }
SLUG        = "geometry"  #@param {type:"string"}
TITLE       = "The Geometry of X"  #@param {type:"string"}
SUBTITLE    = ""  #@param {type:"string"}
REPO        = "zistgah/geometry"  #@param {type:"string"}
DESCRIPTION = "<p>One paragraph. This becomes the Zenodo abstract \u2014 an empty one fails doctor.</p>"  #@param {type:"string"}
KEYWORDS    = "geometry; ontology"  #@param {type:"string"}

import sys, json, importlib.util
spec = importlib.util.spec_from_file_location("matba", "matba.py")
m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
m.HOME = os.environ["MATBA_HOME"]

if not os.path.exists(os.path.join(m.HOME, SLUG, "project.json")):
    m.cmd_new(SLUG, TITLE, SUBTITLE, REPO)
prj = m.load(SLUG)
prj.update(title=TITLE, subtitle=SUBTITLE, repo=REPO, description=DESCRIPTION,
           keywords=[k.strip() for k in KEYWORDS.split(";") if k.strip()])
m.save(prj)
print(json.dumps(m.cmd_intake(SLUG, POSTER_DIR), indent=2))

In [ ]:
#@title 4 · Fill the plate table, then pick the cover  { display-mode: "form" }
# One row per plate. Tab or | separated:
#   file-or-slug | title | subtitle | part | topics(;) | lead paragraph | explanation
BULK = '''
plate-01 | The First Plate | a subtitle | Part One | alpha; beta | The lead paragraph. | Why it matters.
plate-02 | The Second Plate |  | Part One | gamma | The lead paragraph. |
'''  #@param {type:"string"}
COVER = "cover.png"  #@param {type:"string"}

print(m.cmd_bulk(SLUG, BULK), "rows matched")
try:
    m.cmd_set(SLUG, COVER, cover=True); print("cover:", COVER)
except SystemExit as e:
    print("cover not set:", e)

for i, p in enumerate(m.load(SLUG)["plates"], 1):
    print("%2d  %-26s %-26s %-14s %s" % (i, p["file"], p["slug"], p["part"],
                                         "COVER" if p.get("cover") else p["title"][:38]))

In [ ]:
#@title 5 · Doctor, then build  { display-mode: "form" }
fails = m.doctor(m.load(SLUG))
if fails:
    print("DOCTOR FAILS — fix these before building:")
    for f in fails: print("  FAIL", f)
else:
    print("doctor: no failures\n")
    import json; print(json.dumps(m.cmd_build(SLUG), indent=2))

In [ ]:
#@title 6 · Optional: install gh and misty for the full path  { display-mode: "form" }
!type -p gh >/dev/null || (curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg \
   | sudo dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg >/dev/null 2>&1 \
 && echo "deb [signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" \
   | sudo tee /etc/apt/sources.list.d/github-cli.list >/dev/null \
 && sudo apt-get update -qq && sudo apt-get install -y -qq gh)
!pip -q install misty-doi
!type -p gh && gh --version | head -1
!type -p misty && misty --version

In [ ]:
#@title 7 · Authenticate  { display-mode: "form" }
# Tokens by path or prompt. Never pasted into a cell you might commit.
from getpass import getpass
import os, pathlib

# GitHub — a token with repo scope
if not os.environ.get("GH_TOKEN"):
    os.environ["GH_TOKEN"] = getpass("GitHub token (repo scope): ")
!echo "$GH_TOKEN" | gh auth login --with-token && gh auth status

# Zenodo — written to a file, which is what the seeder reads
zp = "/content/zenodo_token"
if not os.path.exists(zp):
    pathlib.Path(zp).write_text(getpass("Zenodo token: ").strip())
    os.chmod(zp, 0o600)
os.environ["ZENODO_TOKEN_PATH"] = zp
print("zenodo token at", zp)

In [ ]:
#@title 8 · Stage — writes nothing upstream  { display-mode: "form" }
p = m.cmd_run(SLUG, "stage")
for line in p.stdout: print(line, end="")
print("\nexit:", p.wait())

In [ ]:
#@title 9 · Push — creates the repo if absent, seals, pushes, verifies  { display-mode: "form" }
word = input('Type the gate word exactly ("PUSH seed") or anything else to abort: ')
p = m.cmd_run(SLUG, "push", word=word)
for line in p.stdout: print(line, end="")
print("\nexit:", p.wait(), "  (3 = gate refused, nothing done)")

In [ ]:
#@title 10 · Mint — permanent  { display-mode: "form" }
OVERRIDE_REHEARSAL = True  #@param {type:"boolean"}
print("MINT is permanent. It publishes a DOI under your Zenodo account.")
print('The push gate is asked first, then the mint gate ("MINT %s").' % SLUG)
w1 = input('PUSH gate word: ')
w2 = input('MINT gate word: ')
if w2.strip() != "MINT " + SLUG:
    print("mint word does not match — not running.")
else:
    p = m.cmd_run(SLUG, "mint", word=w1, override=OVERRIDE_REHEARSAL)
    for line in p.stdout: print(line, end="")
    print("\nexit:", p.wait())

---

### What just happened

`matba` wrote **one generic seeder** for your book and drove it. It did not reimplement `git`,
`gh` or `misty` — the same engine that minted `zenodo.21917807`, `.21928710` and `.21930508`.

The DOI is recorded back into `config/book.config.json` **and** `metadata/misty.json`, the manifest
is re-sealed and re-verified, and the repo is pushed again with that record. If any of that failed
you will see it above; nothing is reported green that did not run.

**Colab caveat:** the VM is ephemeral. Keep `MATBA_HOME` on Drive
(`os.environ["MATBA_HOME"] = "/content/drive/MyDrive/matba-projects"` in cell 1) if you want the
project to survive the session.